# 07 · Comparativa final

**Taller B5-T1 · Generación de datos financieros sinteticos**

> **Notebook pendiente de asignar.** Pietro y Alonso deben repartirse entre
> ellos este notebook, el del otro generador y el de la comparativa final.
> Conviene decidirlo antes de empezar para no duplicar trabajo.
>
> Los bloques marcados como **PENDIENTE** son los que hay que completar. El
> resto, incluida la carga de datos y el guardado de resultados, ya esta
> resuelto y no conviene modificarlo: es lo que garantiza que los resultados de
> los cuatro generadores sean comparables entre si.
>
> Antes de empezar, leer `docs/GUIA_EQUIPO.md` y usar
> `04_generador_cgan.ipynb` como referencia de estructura y de estilo.

>
> Este notebook depende de que los cuatro generadores hayan dejado su tabla en
> `results/tablas/`. Es el último en ejecutarse, y de el salen las figuras que se
> proyectan en la defensa: conviene que se lean bien de lejos.

## Preparación del entorno

Se fija el backend de cómputo, se añade el código común del proyecto a la ruta
de importación y se aplica el estilo gráfico compartido. El bloque funciona sin
cambios tanto en una instalación local como en Colab.

In [ ]:
import os
import sys

# Keras 3 se ejecuta sobre PyTorch: la versión de Python empleada no dispone de
# TensorFlow y la API de capas y modelos es idéntica en ambos backends.
os.environ["KERAS_BACKEND"] = "torch"

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    !pip install -q yfinance keras torch
    # En Colab se asume que el repositorio está clonado en el directorio actual.
    RAIZ = "/content/B5-T1"
else:
    RAIZ = os.path.dirname(os.getcwd())

if os.path.join(RAIZ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(RAIZ, "src"))

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from miax_b5t1 import config, datos, experimento, graficos, modelo

graficos.aplicar_estilo()
config.asegurar_directorios()

print(f"Keras {keras.__version__} sobre backend {keras.backend.backend()}")
print(f"Raíz del proyecto: {RAIZ}")

## 1. Reunion de resultados

Se cargan las tablas de todos los modelos. Cada una fue generada por el mismo
protocolo de evaluación, de modo que las filas son directamente comparables.

Si algún generador aún no ha terminado, la función avisa y continúa con los
disponibles, lo que permite ir preparando el notebook sin esperar a que esten
todos.

In [ ]:
MODELOS = ["baseline", "ruido", "cgan", "cvae", "diffusion"]

resultados = experimento.cargar_resultados(MODELOS)
resumen = experimento.resumir(resultados)

print(f"Configuraciones evaluadas: {len(resultados)}")
print(f"Modelos disponibles: {sorted(resultados['modelo'].unique())}")
display(resumen.round(4))

## 2. Figura principal

La figura central del trabajo enfrenta la cantidad de datos sinteticos añadidos
con la calidad alcanzada por el clasificador, con una curva por modelo
generativo. Conviene representarla sobre test y sobre validación, ya que el
notebook 02 mostro que ambas particiones se comportan de forma distinta.

In [ ]:
graficos.curva_ratios(
    resumen, metrica="pr_auc_media", desviación="pr_auc_std",
    titulo="Efecto del dato sintetico sobre el test",
    nombre_fichero="07_comparativa_test")
plt.show()

graficos.curva_ratios(
    resumen, metrica="pr_auc_val_media", desviación="pr_auc_val_std",
    titulo="Efecto del dato sintetico sobre validación",
    nombre_fichero="07_comparativa_validación")
plt.show()

## 3. Significación de las diferencias

**PENDIENTE:** comprobar si las diferencias observadas superan la variabilidad
entre semillas.

Es el punto más importante del notebook. Con cinco semillas por configuración y
un test que cubre un único régimen de mercado, muchas diferencias aparentes no
resisten un examen mínimo. Afirmar que un generador mejora a otro sin comprobarlo
es el error más fácil de señalar en la defensa.

Orientaciones:

- Comparar la diferencia entre medias con la desviación tipica entre semillas.
- Un contraste de medias entre la configuración sin sinteticos y la mejor de cada
  modelo basta para el proposito.
- Presentar el resultado en una tabla con la mejora estimada y su margen de
  incertidumbre.

In [ ]:
# PENDIENTE: contraste de las diferencias entre modelos.

## 4. Tabla resumen

**PENDIENTE:** construir la tabla final del informe.

Una fila por modelo generativo, con el PR-AUC sin sinteticos, el mejor PR-AUC
alcanzado, el ratio en que se alcanza y la variación respecto a la referencia.
Conviene incluir también el resultado en validación.

Guardarla en `results/tablas/` con `to_csv`, ya que el enunciado pide que el
código genere todas las tablas reportadas.

In [ ]:
# PENDIENTE: tabla resumen y guardado en results/tablas/comparativa_final.csv

## 5. Calidad de las muestras frente a utilidad

**PENDIENTE:** relacionar el realismo de cada generador con su efecto real.

Los notebooks de cada generador miden hasta que punto las muestras reproducen las
propiedades de los datos reales. Merece la pena comprobar si el generador más
realista es también el más útil para el clasificador, porque no tiene por que
serlo: un generador puede reproducir fielmente la distribución y no aportar
ninguna configuración que el clasificador no tuviera ya.

Los ficheros `models/sinteticos_<modelo>.npz` se cargan con
`experimento.cargar_sinteticos`.

In [ ]:
# PENDIENTE: comparación entre realismo de las muestras y efecto sobre el modelo.

## Conclusiones del proyecto

**PENDIENTE:** este es el texto que sostiene la presentación. Conviene que
responda a:

1. Cual de los cuatro generadores produce las muestras más fieles a los datos
   reales.
2. Si añadir datos sinteticos mejora el clasificador, en que medida y a partir de
   que proporción.
3. Si algun modelo neuronal supera al generador por ruido, que es el nivel mínimo
   exigible, y si su complejidad queda justificada.
4. Que ocurre cuando la proporción de sinteticos es muy alta.
5. Que conclusión general se extrae sobre el uso de datos sinteticos en este
   problema concreto, teniendo en cuenta lo que mostro el notebook 02: que la
   limitación no es el volumen de datos sino la variedad de episodios de estrés.

Si la conclusión es que los datos sinteticos no ayudan, se dice con claridad y se
explica por que. Un resultado negativo bien argumentado y correctamente medido
vale más que una mejora aparente que no resiste el contraste con la variabilidad
entre semillas.